In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from ast import literal_eval
import pybirewirex

In [118]:
modules_df = pd.read_csv('/home/lnemati/pathway_crosstalk/results/comparison/modules_all_gene_all_tissues.csv', index_col=0)

ccc = pd.read_csv('/home/lnemati/pathway_crosstalk/data/interactions/ccc.csv')
ccc['all_genes'] = ccc['all_genes'].apply(literal_eval)

cci_genes = list(set(ccc['all_genes'].sum()))

t_tissues = list(tissues_df.query('condition == "tumor"')['subtissue'])
n_tissues = list(tissues_df.query('condition == "normal"')['subtissue'])

t_modules = modules_df[[t for t in t_tissues if t in modules_df.columns]]
n_modules = modules_df[[n for n in n_tissues if n in modules_df.columns]]

# Subset to only CCI genes
t_modules = t_modules.loc[[g for g in cci_genes if g in t_modules.index]]
n_modules = n_modules.loc[[g for g in cci_genes if g in n_modules.index]]

In [17]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# -------------------------
# FORCE FLOAT MATRICES
# -------------------------
t_mat = t_modules.to_numpy(dtype=float)
n_mat = n_modules.to_numpy(dtype=float)

gene_to_t_idx = {g: i for i, g in enumerate(t_modules.index)}
gene_to_n_idx = {g: i for i, g in enumerate(n_modules.index)}

# -------------------------
# PRECOMPUTE CCI INDEX LISTS (VERY IMPORTANT SPEEDUP)
# -------------------------
cci_t_idx = []
cci_n_idx = []

for genes in ccc['all_genes']:
    genes = list(genes)

    t_idx = [gene_to_t_idx[g] for g in genes if g in gene_to_t_idx]
    n_idx = [gene_to_n_idx[g] for g in genes if g in gene_to_n_idx]

    if len(t_idx) == len(genes):
        cci_t_idx.append(np.array(t_idx))
    else:
        cci_t_idx.append(None)

    if len(n_idx) == len(genes):
        cci_n_idx.append(np.array(n_idx))
    else:
        cci_n_idx.append(None)

# -------------------------
# FAST FUNCTION
# -------------------------
def has_same_module(mat, row_idx):
    sub = mat[row_idx, :]

    valid_cols = ~np.isnan(sub).any(axis=0)
    if not np.any(valid_cols):
        return False

    sub = sub[:, valid_cols]
    return np.any(np.ptp(sub, axis=0) == 0)

# -------------------------
# OBSERVED
# -------------------------
obs_t = 0
obs_n = 0
obs_both = 0

for i in range(len(cci_t_idx)):

    t_idx = cci_t_idx[i]
    n_idx = cci_n_idx[i]

    t_ok = False
    n_ok = False

    if t_idx is not None:
        t_ok = has_same_module(t_mat, t_idx)

    if n_idx is not None:
        n_ok = has_same_module(n_mat, n_idx)

    obs_t += t_ok
    obs_n += n_ok
    obs_both += (t_ok and n_ok)

# -------------------------
# PERMUTATION (FASTER + LESS PYTHON OVERHEAD)
# -------------------------
n_perm = 1000
rng = np.random.default_rng()

perm_t = np.zeros(n_perm)
perm_n = np.zeros(n_perm)
perm_both = np.zeros(n_perm)

n_genes, n_tissues = t_mat.shape
n_genes_n, n_tissues_n = n_mat.shape

for p in tqdm(range(n_perm), desc="Permutations"):

    # ---- shuffle tumor (vectorized column-wise)
    t_perm = t_mat.copy()
    mask_t = ~np.isnan(t_perm)

    for j in range(n_tissues):
        vals = t_perm[mask_t[:, j], j]
        rng.shuffle(vals)
        t_perm[mask_t[:, j], j] = vals

    # ---- shuffle normal
    n_perm_mat = n_mat.copy()
    mask_n = ~np.isnan(n_perm_mat)

    for j in range(n_tissues_n):
        vals = n_perm_mat[mask_n[:, j], j]
        rng.shuffle(vals)
        n_perm_mat[mask_n[:, j], j] = vals

    # ---- evaluate CCIs (NO dict lookups, NO gene loops)
    ct = 0
    cn = 0
    cb = 0

    for i in range(len(cci_t_idx)):

        t_idx = cci_t_idx[i]
        n_idx = cci_n_idx[i]

        t_ok = False
        n_ok = False

        if t_idx is not None:
            t_ok = has_same_module(t_perm, t_idx)

        if n_idx is not None:
            n_ok = has_same_module(n_perm_mat, n_idx)

        ct += t_ok
        cn += n_ok
        cb += (t_ok and n_ok)

    perm_t[p] = ct
    perm_n[p] = cn
    perm_both[p] = cb

# -------------------------
# P-VALUES
# -------------------------
p_t = (np.sum(perm_t >= obs_t) + 1) / (n_perm + 1)
p_n = (np.sum(perm_n >= obs_n) + 1) / (n_perm + 1)
p_both = (np.sum(perm_both >= obs_both) + 1) / (n_perm + 1)

print(f"Tumor: {obs_t} | Expected: {np.mean(perm_t):.2f} ± {np.std(perm_t):.2f} | p={(np.sum(perm_t >= obs_t)+1)/(n_perm+1):.3g}")
print(f"Normal: {obs_n} | Expected: {np.mean(perm_n):.2f} ± {np.std(perm_n):.2f} | p={(np.sum(perm_n >= obs_n)+1)/(n_perm+1):.3g}")
print(f"Both: {obs_both} | Expected: {np.mean(perm_both):.2f} ± {np.std(perm_both):.2f} | p={(np.sum(perm_both >= obs_both)+1)/(n_perm+1):.3g}")

Permutations: 100%|██████████| 1000/1000 [02:13<00:00,  7.48it/s]

Tumor: 2301 | Expected: 2205.08 ± 34.47 | p=0.005
Normal: 2794 | Expected: 2623.75 ± 44.05 | p=0.000999
Both: 2065 | Expected: 2007.16 ± 29.76 | p=0.028


In [115]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# -------------------------
# FORCE FLOAT MATRICES
# -------------------------
t_mat = t_modules.to_numpy(dtype=float)
n_mat = n_modules.to_numpy(dtype=float)

gene_to_t_idx = {g: i for i, g in enumerate(t_modules.index)}
gene_to_n_idx = {g: i for i, g in enumerate(n_modules.index)}

# -------------------------
# FAST FUNCTION
# -------------------------
def has_same_module(mat, row_idx):
    sub = mat[row_idx, :]

    valid_cols = ~np.isnan(sub).any(axis=0)
    if not np.any(valid_cols):
        return False

    sub = sub[:, valid_cols]
    return np.any(np.ptp(sub, axis=0) == 0)

# -------------------------
# OBSERVED VALUES
# -------------------------
results = []

for genes in ccc['all_genes']:

    genes = list(genes)

    t_idx = [gene_to_t_idx[g] for g in genes if g in gene_to_t_idx]
    n_idx = [gene_to_n_idx[g] for g in genes if g in gene_to_n_idx]

    t_same_module = False
    n_same_module = False

    if len(t_idx) == len(genes):
        t_same_module = has_same_module(t_mat, np.array(t_idx))

    if len(n_idx) == len(genes):
        n_same_module = has_same_module(n_mat, np.array(n_idx))

    results.append({
        "genes": genes,
        "tumor_same_module": t_same_module,
        "normal_same_module": n_same_module
    })

results_df = pd.DataFrame(results)

obs_N_t = results_df['tumor_same_module'].sum()
obs_N_n = results_df['normal_same_module'].sum()
obs_N_both = (results_df['tumor_same_module'] &
              results_df['normal_same_module']).sum()

print("Observed:")
print("Tumor:", obs_N_t)
print("Normal:", obs_N_n)
print("Both:", obs_N_both)

# -------------------------
# PERMUTATION TEST
# -------------------------
n_perm = 100

perm_N_t = np.zeros(n_perm)
perm_N_n = np.zeros(n_perm)
perm_N_both = np.zeros(n_perm)

rng = np.random.default_rng()

for p in tqdm(range(n_perm), desc="Permutations"):

    # shuffle each column independently
    t_perm = np.apply_along_axis(rng.permutation, 0, t_mat)
    n_perm_mat = np.apply_along_axis(rng.permutation, 0, n_mat)

    perm_t = []
    perm_n = []

    for genes in ccc['all_genes']:

        genes = list(genes)

        t_idx = [gene_to_t_idx[g] for g in genes if g in gene_to_t_idx]
        n_idx = [gene_to_n_idx[g] for g in genes if g in gene_to_n_idx]

        t_same_module = False
        n_same_module = False

        if len(t_idx) == len(genes):
            t_same_module = has_same_module(t_perm, np.array(t_idx))

        if len(n_idx) == len(genes):
            n_same_module = has_same_module(n_perm_mat, np.array(n_idx))

        perm_t.append(t_same_module)
        perm_n.append(n_same_module)

    perm_t = np.array(perm_t)
    perm_n = np.array(perm_n)

    perm_N_t[p] = perm_t.sum()
    perm_N_n[p] = perm_n.sum()
    perm_N_both[p] = (perm_t & perm_n).sum()

# -------------------------
# P-VALUES
# -------------------------
p_t = (np.sum(perm_N_t >= obs_N_t) + 1) / (n_perm + 1)
p_n = (np.sum(perm_N_n >= obs_N_n) + 1) / (n_perm + 1)
p_both = (np.sum(perm_N_both >= obs_N_both) + 1) / (n_perm + 1)

print("\nEmpirical p-values:")
print("Tumor:", p_t)
print("Normal:", p_n)
print("Both:", p_both)

Observed:
Tumor: 2301
Normal: 2794
Both: 2065


Permutations: 100%|██████████| 100/100 [00:18<00:00,  5.30it/s]


Empirical p-values:
Tumor: 0.009900990099009901
Normal: 0.009900990099009901
Both: 0.009900990099009901


In [116]:
perm_N_t

array([1010., 1048., 1032.,  997., 1010.,  960., 1025., 1022., 1069.,
       1056., 1021., 1047., 1025., 1010., 1060.,  997.,  993., 1049.,
       1030., 1055., 1022., 1029., 1048., 1055., 1084.,  998., 1032.,
       1038., 1039., 1022., 1039., 1028., 1003., 1056., 1010., 1001.,
       1027., 1030., 1038.,  987., 1021., 1004., 1027.,  987., 1021.,
        999., 1063., 1024., 1090.,  982., 1045., 1019., 1058., 1003.,
       1025., 1016.,  983., 1061., 1024., 1038., 1031., 1021., 1025.,
       1053., 1017., 1053., 1025.,  990., 1022., 1058., 1039.,  998.,
       1034., 1038., 1054., 1052., 1064., 1022., 1012., 1055.,  993.,
       1021., 1021., 1037., 1035.,  973., 1038., 1016.,  995., 1016.,
       1044., 1047., 1048., 1040., 1047., 1032., 1017., 1036., 1056.,
       1022.])

In [117]:
perm_N_n

array([1246., 1224., 1204., 1228., 1218., 1291., 1242., 1181., 1230.,
       1193., 1214., 1220., 1252., 1251., 1187., 1258., 1241., 1254.,
       1197., 1272., 1212., 1217., 1188., 1210., 1236., 1190., 1265.,
       1245., 1291., 1264., 1243., 1202., 1215., 1202., 1246., 1269.,
       1230., 1263., 1236., 1255., 1203., 1265., 1209., 1240., 1229.,
       1228., 1253., 1217., 1203., 1248., 1198., 1274., 1198., 1198.,
       1216., 1241., 1231., 1219., 1258., 1225., 1201., 1315., 1198.,
       1252., 1229., 1232., 1282., 1316., 1234., 1227., 1278., 1195.,
       1211., 1192., 1262., 1247., 1222., 1271., 1206., 1198., 1275.,
       1237., 1249., 1229., 1244., 1250., 1263., 1207., 1210., 1265.,
       1245., 1237., 1233., 1190., 1238., 1221., 1214., 1212., 1235.,
       1249.])

In [111]:
perm_N_both

array([2369., 2399., 2333., 2165., 2430., 2293., 2366., 2431., 2206.,
       2378., 2401., 2270., 2372., 2369., 2272., 2369., 2243., 2314.,
       2197., 2275., 2420., 2309., 2397., 2284., 2322., 2367., 2251.,
       2309., 2416., 2295., 2385., 2292., 2147., 2324., 2393., 2277.,
       2372., 2241., 2447., 2312., 2238., 2339., 2316., 2266., 2333.,
       2301., 2375., 2293., 2318., 2388., 2214., 2394., 2198., 2383.,
       2336., 2324., 2305., 2411., 2272., 2464., 2340., 2283., 2357.,
       2257., 2267., 2344., 2419., 2395., 2323., 2209., 2366., 2330.,
       2386., 2397., 2275., 2267., 2317., 2319., 2261., 2340., 2321.,
       2433., 2356., 2376., 2464., 2341., 2262., 2261., 2339., 2269.,
       2375., 2361., 2497., 2292., 2340., 2343., 2300., 2178., 2163.,
       2360.])